In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType
import os

spark = SparkSession.builder \
    .appName('NetworkIntrusionETL_Transform') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

STAGING = os.path.abspath(os.path.join(os.getcwd(), '..', 'Data', 'staging')).replace('\\', '/')
print('Staging dir:', STAGING)

Staging dir: d:/Projects/Big Data/BG  Assignmnet/Assignmnet/Data/staging


In [2]:
# ── Load staged raw parquet ───────────────────────────────────────────────────
cicids_raw   = spark.read.parquet(STAGING + '/cicids_raw')
unsw_raw     = spark.read.parquet(STAGING + '/unsw_train_raw')
conn_log_raw = spark.read.parquet(STAGING + '/conn_log_raw')

print('Loaded all staged sources')

Loaded all staged sources


In [3]:
# ── Transform CICIDS2017 ──────────────────────────────────────────────────────
def clean_col(c):
    return c.strip().lower().replace(' ', '_').replace('/', '_').replace('-', '_')

cicids = cicids_raw
for col in cicids.columns:
    cicids = cicids.withColumnRenamed(col, clean_col(col))

cicids = cicids.filter(F.col('attack_type').isNotNull())

num_cols = [f.name for f in cicids.schema.fields
            if str(f.dataType) in ('DoubleType()', 'FloatType()', 'IntegerType()', 'LongType()')]
for c in num_cols:
    cicids = cicids.withColumn(c,
        F.when(F.col(c).isin([float('inf'), float('-inf')]), None).otherwise(F.col(c)))
cicids = cicids.fillna(0, subset=num_cols)

cicids = cicids \
    .withColumn('source_dataset', F.lit('CICIDS2017')) \
    .withColumn('is_attack', F.when(F.lower(F.col('attack_type')) == 'benign', 0).otherwise(1))

cicids_final = cicids.select(
    F.col('destination_port').cast(IntegerType()).alias('dst_port'),
    F.col('flow_duration').cast(DoubleType()).alias('duration'),
    F.col('total_fwd_packets').cast(IntegerType()).alias('src_pkts'),
    F.col('flow_bytes_s').cast(DoubleType()).alias('flow_bytes_per_sec'),
    F.col('flow_packets_s').cast(DoubleType()).alias('flow_pkts_per_sec'),
    F.col('packet_length_mean').cast(DoubleType()).alias('mean_pkt_len'),
    F.col('packet_length_std').cast(DoubleType()).alias('std_pkt_len'),
    F.col('fin_flag_count').cast(IntegerType()).alias('fin_flag'),
    F.col('psh_flag_count').cast(IntegerType()).alias('psh_flag'),
    F.col('ack_flag_count').cast(IntegerType()).alias('ack_flag'),
    F.col('attack_type').alias('attack_category'),
    F.col('is_attack').cast(IntegerType()),
    F.col('source_dataset')
)

print('CICIDS rows:', cicids_final.count())
cicids_final.show(3)

CICIDS rows: 2520751
+--------+-----------+--------+------------------+-----------------+------------+-----------+--------+--------+--------+---------------+---------+--------------+
|dst_port|   duration|src_pkts|flow_bytes_per_sec|flow_pkts_per_sec|mean_pkt_len|std_pkt_len|fin_flag|psh_flag|ack_flag|attack_category|is_attack|source_dataset|
+--------+-----------+--------+------------------+-----------------+------------+-----------+--------+--------+--------+---------------+---------+--------------+
|     443|7.0098634E7|      14|       125.2235529|       0.44223401|    274.3125|560.9237449|       0|       1|       0| Normal Traffic|        1|    CICIDS2017|
|     443|   126179.0|       9|       39150.73031|      134.7292339| 274.4444444|501.2102086|       0|       1|       0| Normal Traffic|        1|    CICIDS2017|
|      53|1.6285684E7|       2|       23.57899122|      0.245614492|        87.6| 54.6699186|       0|       0|       0| Normal Traffic|        1|    CICIDS2017|
+------

In [4]:
# ── Transform UNSW-NB15 ───────────────────────────────────────────────────────
unsw = unsw_raw
for c in ['proto', 'service', 'state', 'attack_cat']:
    if c in unsw.columns:
        unsw = unsw.withColumn(c, F.col(c).cast(StringType()))

unsw_num = [f.name for f in unsw.schema.fields
            if str(f.dataType) in ('FloatType()', 'DoubleType()', 'IntegerType()', 'LongType()', 'ShortType()')]
unsw = unsw.fillna(0, subset=unsw_num)
unsw = unsw.fillna('Unknown', subset=['attack_cat', 'service'])
unsw = unsw.withColumn('source_dataset', F.lit('UNSW_NB15'))

unsw_final = unsw.select(
    F.lit(None).cast(IntegerType()).alias('dst_port'),
    F.col('dur').cast(DoubleType()).alias('duration'),
    F.col('spkts').cast(IntegerType()).alias('src_pkts'),
    F.col('sload').cast(DoubleType()).alias('flow_bytes_per_sec'),
    F.col('rate').cast(DoubleType()).alias('flow_pkts_per_sec'),
    F.col('smean').cast(DoubleType()).alias('mean_pkt_len'),
    F.lit(None).cast(DoubleType()).alias('std_pkt_len'),
    F.lit(None).cast(IntegerType()).alias('fin_flag'),
    F.lit(None).cast(IntegerType()).alias('psh_flag'),
    F.lit(None).cast(IntegerType()).alias('ack_flag'),
    F.col('attack_cat').alias('attack_category'),
    F.col('label').cast(IntegerType()).alias('is_attack'),
    F.col('source_dataset')
)

print('UNSW rows:', unsw_final.count())
unsw_final.show(3)

UNSW rows: 175341
+--------+-------------------+--------+------------------+------------------+------------+-----------+--------+--------+--------+---------------+---------+--------------+
|dst_port|           duration|src_pkts|flow_bytes_per_sec| flow_pkts_per_sec|mean_pkt_len|std_pkt_len|fin_flag|psh_flag|ack_flag|attack_category|is_attack|source_dataset|
+--------+-------------------+--------+------------------+------------------+------------+-----------+--------+--------+--------+---------------+---------+--------------+
|    NULL|0.12147799879312515|       6|  14158.9423828125| 74.08748626708984|        43.0|       NULL|    NULL|    NULL|    NULL|         Normal|        0|     UNSW_NB15|
|    NULL| 0.6499019861221313|      14|   8395.1123046875| 78.47337341308594|        52.0|       NULL|    NULL|    NULL|    NULL|         Normal|        0|     UNSW_NB15|
|    NULL| 1.6231290102005005|       8|1572.2718505859375|14.170161247253418|        46.0|       NULL|    NULL|    NULL|    NUL

In [5]:
# ── Transform Zeek conn.log ───────────────────────────────────────────────────
conn = conn_log_raw

for c in ['duration', 'orig_bytes', 'resp_bytes', 'missed_bytes',
          'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes']:
    conn = conn.withColumn(c,
        F.when(F.col(c).cast(StringType()) == '-', None)
         .otherwise(F.col(c).cast(DoubleType())))

for c in ['service', 'proto', 'conn_state']:
    conn = conn.withColumn(c,
        F.when(F.col(c) == '-', None).otherwise(F.col(c).cast(StringType())))

conn = conn.fillna(0.0, subset=['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts'])
conn = conn.fillna('Unknown', subset=['service', 'proto'])
conn = conn.withColumn('source_dataset', F.lit('ZEEK_CONN'))

conn = conn.withColumn('flow_bytes_per_sec',
    F.when(F.col('duration') > 0,
           (F.col('orig_bytes') + F.col('resp_bytes')) / F.col('duration')
    ).otherwise(0.0))

conn = conn.withColumn('flow_pkts_per_sec',
    F.when(F.col('duration') > 0,
           (F.col('orig_pkts') + F.col('resp_pkts')) / F.col('duration')
    ).otherwise(0.0))

conn = conn.withColumn('mean_pkt_len',
    F.when((F.col('orig_pkts') + F.col('resp_pkts')) > 0,
           (F.col('orig_ip_bytes') + F.col('resp_ip_bytes')) /
           (F.col('orig_pkts') + F.col('resp_pkts'))
    ).otherwise(0.0))

conn_final = conn.select(
    F.col('resp_p').cast(IntegerType()).alias('dst_port'),
    F.col('duration').cast(DoubleType()),
    F.col('orig_pkts').cast(IntegerType()).alias('src_pkts'),
    F.col('flow_bytes_per_sec').cast(DoubleType()),
    F.col('flow_pkts_per_sec').cast(DoubleType()),
    F.col('mean_pkt_len').cast(DoubleType()),
    F.lit(None).cast(DoubleType()).alias('std_pkt_len'),
    F.lit(None).cast(IntegerType()).alias('fin_flag'),
    F.lit(None).cast(IntegerType()).alias('psh_flag'),
    F.lit(None).cast(IntegerType()).alias('ack_flag'),
    F.col('service').alias('attack_category'),
    F.lit(0).cast(IntegerType()).alias('is_attack'),
    F.col('source_dataset')
)

print('Zeek rows:', conn_final.count())
conn_final.show(3)

Zeek rows: 36394
+--------+--------+--------+------------------+------------------+------------------+-----------+--------+--------+--------+---------------+---------+--------------+
|dst_port|duration|src_pkts|flow_bytes_per_sec| flow_pkts_per_sec|      mean_pkt_len|std_pkt_len|fin_flag|psh_flag|ack_flag|attack_category|is_attack|source_dataset|
+--------+--------+--------+------------------+------------------+------------------+-----------+--------+--------+--------+---------------+---------+--------------+
|   16412|0.021641|      44| 572894.0437133219|4066.3555288572616|333.95454545454544|       NULL|    NULL|    NULL|    NULL|        Unknown|        0|     ZEEK_CONN|
|      80|0.192316|      20| 131616.7141579484| 322.3860729216498|  868.516129032258|       NULL|    NULL|    NULL|    NULL|        Unknown|        0|     ZEEK_CONN|
|   39254| 0.06546|      86| 587045.5239841124|2597.0058050717994|504.18823529411765|       NULL|    NULL|    NULL|    NULL|        Unknown|        0|   

In [6]:
# ── Union all three sources ───────────────────────────────────────────────────
unified = cicids_final.unionByName(unsw_final).unionByName(conn_final)

print('Unified total rows:', unified.count())
unified.groupBy('source_dataset', 'is_attack').count().orderBy('source_dataset', 'is_attack').show()

unified.write.mode('overwrite').parquet(STAGING + '/unified_transformed')
print('Saved unified_transformed')

Unified total rows: 2732486
+--------------+---------+-------+
|source_dataset|is_attack|  count|
+--------------+---------+-------+
|    CICIDS2017|        1|2520751|
|     UNSW_NB15|        0|  56000|
|     UNSW_NB15|        1| 119341|
|     ZEEK_CONN|        0|  36394|
+--------------+---------+-------+

Saved unified_transformed


In [7]:
# ── Aggregated summary ────────────────────────────────────────────────────────
summary = unified.groupBy('source_dataset', 'attack_category', 'is_attack').agg(
    F.count('*').alias('record_count'),
    F.avg('duration').alias('avg_duration'),
    F.avg('flow_bytes_per_sec').alias('avg_flow_bytes_per_sec'),
    F.avg('flow_pkts_per_sec').alias('avg_flow_pkts_per_sec'),
    F.avg('mean_pkt_len').alias('avg_mean_pkt_len'),
    F.avg('src_pkts').alias('avg_src_pkts')
)

summary.write.mode('overwrite').parquet(STAGING + '/summary_agg')
print('Saved summary_agg')
summary.show(10)

Saved summary_agg
+--------------+---------------+---------+------------+--------------------+----------------------+---------------------+------------------+------------------+
|source_dataset|attack_category|is_attack|record_count|        avg_duration|avg_flow_bytes_per_sec|avg_flow_pkts_per_sec|  avg_mean_pkt_len|      avg_src_pkts|
+--------------+---------------+---------+------------+--------------------+----------------------+---------------------+------------------+------------------+
|    CICIDS2017|    Brute Force|        1|        9150|   7891160.129726776|    25015.564103340417|    5818.590606685821| 35.17054496613805|11.207978142076502|
|    CICIDS2017| Normal Traffic|        1|     2095057|1.2160238432630712E7|    1675795.0940971514|    53557.47818858114|115.89230088161271| 11.37581411866121|
|    CICIDS2017|    Web Attacks|        1|        2143|   6646525.880074661|    160.63998017767622|   2406.2949946987087|26.009516723639774|11.117125524965003|
|    CICIDS2017|      